# 实践 3.1  遗传力、重复数与BLUP收缩的虚拟试验

《统计遗传与智能育种学》第三章 · 电子资源 `03b_heritability_shrinkage.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：03b_heritability_shrinkage.ipynb。实践先从N(0,1)抽取一次真实育种值（true breeding value, TBV）并固定，在总体均值0、遗传方差1已知的条件下，按目标遗传力设置残差方差、改变重复数生成表型；同时报告目标遗传力与模拟样本遗传方差比（realized_h2），比较原始材料均值与收缩预测的均方误差（mean squared error, MSE），并展示低遗传力、低重复条件下更明显的收缩。随后在h²=0.35下比较1、2、4、8个重复的理论可靠度，以数值结果说明重复数对估计精度的影响。

核心关系：σ²e = σ²g(1−h²)/h²  
比较：单次观测、重复均值、收缩预测  
验收：固定随机种子；报告模拟样本遗传方差比、预测相关与均方误差。

## 输入、完整代码、实际输出与解释

输入与可复现条件：固定一组真实遗传值，目标遗传力0.2/0.5/0.8，重复数1/2/4；SEED=301；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=301
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 301


In [2]:
n=180; u=rng.normal(0,1,n)
def experiment(h2,reps):
    ve=(1-h2)/h2
    Y=u[:,None]+rng.normal(0,np.sqrt(ve),(n,reps)); mean=Y.mean(axis=1)
    reliability=1/(1+ve/reps); blup=reliability*mean
    vg=np.var(u,ddof=1); ve_real=np.var((Y-u[:,None]).ravel(),ddof=1); realized_h2=vg/(vg+ve_real)
    return {'target_h2':h2,'realized_h2':realized_h2,'reps':reps,'shrink':reliability,'MSE_mean':np.mean((mean-u)**2),'MSE_BLUP':np.mean((blup-u)**2),'r_mean_TBV':np.corrcoef(mean,u)[0,1]}
rows=[experiment(h,r) for h in [.2,.5,.8] for r in [1,2,4]]
print(pd.DataFrame(rows).round(4).to_string(index=False))

# 同一目标遗传力下，重复数增加如何提高基因型均值可靠性
h=.35; ve=(1-h)/h
for r in [1,2,4,8]:
 rel=1/(1+ve/r); print('reps',r,'theoretical reliability',round(rel,3))


 target_h2  realized_h2  reps  shrink  MSE_mean  MSE_BLUP  r_mean_TBV
       0.2       0.1781     1  0.2000    4.6234    0.8572      0.3805
       0.2       0.2060     2  0.3333    1.8390    0.7258      0.5303
       0.2       0.2058     4  0.5000    0.9797    0.4648      0.7323
       0.5       0.4852     1  0.5000    1.0535    0.5261      0.6883
       0.5       0.5067     2  0.6667    0.5314    0.3454      0.8078
       0.5       0.4934     4  0.8000    0.2556    0.2054      0.8920
       0.8       0.7862     1  0.8000    0.2712    0.2151      0.8864
       0.8       0.8146     2  0.8889    0.1053    0.1107      0.9460
       0.8       0.8058     4  0.9412    0.0609    0.0564      0.9713
reps 1 theoretical reliability 0.35
reps 2 theoretical reliability 0.519
reps 4 theoretical reliability 0.683
reps 8 theoretical reliability 0.812


结果解释：同一目标遗传力下增加重复通常降低材料均值误差并提高与真实遗传值的相关；BLUP收缩在多数场景降低MSE。realized_h2按遗传与残差的样本方差计算。表中的shrink是当前简化模型的收缩系数；末尾reliability示例固定h²=0.35，仅用于说明重复数增加时基因型均值可靠度的理论变化，不能替代完整混合模型的方差组分估计。

可修改练习：把材料数减半并重复不同SEED，比较realized_h2偏离目标值的幅度；再把重复数扩展到8，观察可靠度增益是否线性。